# Jour 1 : entraîner un premier YOLO, tracé dans MLflow

**Quand ?** Jour 1, une fois les images annotées dans CVAT et la tâche exportée par le chef d'équipe.

**Quoi ?** Ce notebook « à trous » vous guide, étape par étape :

0. vérifier le GPU et installer les bibliothèques ;
1. renseigner vos paramètres (login, squad, membre, version, export CVAT) ;
2. vous connecter à MLflow (via le portail) et vérifier la connexion ;
3. décompresser l'export CVAT et le découper en `train` / `val` (≈ 80 / 20) ;
4. entraîner YOLO **dans un run MLflow** (paramètres, métriques, `best.pt`) ;
5. exporter le modèle en **ONNX** et le ranger dans le même run ;
6. regarder les résultats et quelques prédictions ;
7. enregistrer le modèle dans le registre avec l'alias `champion`.

**Pourquoi ?** Tous vos entraînements sont tracés dans MLflow *dès le premier* : sans suivi, personne ne sait plus quel modèle est en production, avec quelles données ni quels paramètres.

**Prérequis**

- l'archive `.zip` exportée de CVAT (format *Ultralytics YOLO Detection 1.0*), déposée dans Colab (panneau *Fichiers*) ;
- le fichier `identifiants-<login>.json` téléchargé depuis votre profil du portail, déposé au même endroit ;
- un GPU : *Exécution > Modifier le type d'exécution > GPU (T4)*.

> **Les cellules marquées `# TODO` sont à compléter par vous.** Les indices sont dans le texte juste au-dessus.
> **La métrique qui compte le plus est le rappel** (*recall*) : une case oubliée, c'est un créneau perdu.

## 0. GPU et installation

Activez le GPU **avant** d'installer quoi que ce soit : changer de type d'exécution réinitialise la session.

In [ ]:
# Vérification du GPU (doit afficher « Tesla T4 » ou similaire)
!nvidia-smi -L

In [1]:
%pip install -q ultralytics mlflow onnx onnxruntime onnxslim

## 1. Paramètres

Renseignez vos valeurs. Rappel des conventions :

| Élément | Convention | Exemple |
|---|---|---|
| `SQUAD` | nom d'équipe en minuscules, avec des tirets | `team-rocket`, `team-magma`, `team-aqua` |
| expérience MLflow | `flexistaff_<squad>` (commune à toute la squad) | `flexistaff_team-rocket` |
| nom du run | `<membre>_<version>` | `toto_v1` |
| modèle enregistré | `flexistaff_<squad>` | `flexistaff_team-rocket` |

> **TODO** : remplacez les valeurs `"..."` (login, squad, membre). Les valeurs par défaut des autres paramètres conviennent pour ce premier run.

In [7]:
LOGIN = "carole.n"                 # TODO : votre login du portail (le même que pour CVAT)
SQUAD = "team-rocket"                 # TODO : team-rocket, team-magma ou team-aqua
MEMBRE = "carole"                # TODO : votre prénom en minuscules, ex. "toto"
VERSION = "v1"                # version de votre entraînement : v1, puis v2, v3...
DATASET_VERSION = "v1"        # version du jeu de données (à changer si vous réexportez CVAT)

CVAT_ZIP = "cvat_export.zip"  # archive exportée de CVAT, déposée dans le panneau Fichiers
CREDENTIALS = f"identifiants-{LOGIN}.json"   # fichier téléchargé depuis votre profil du portail

DATASET_DIR = "dataset"       # dossier de travail du jeu de données
RATIO_TRAIN = 0.8             # ≈ 80 % des images en train, 20 % en val
SEED = 42                     # graine du tirage : même découpage à chaque exécution

# Déduits des paramètres ci-dessus (ne pas modifier)
EXPERIMENT = f"flexistaff_{SQUAD}"
MODEL_NAME = f"flexistaff_{SQUAD}"
RUN_NAME = f"{MEMBRE}_{VERSION}"
assert "..." not in (LOGIN, SQUAD, MEMBRE), "Complétez LOGIN, SQUAD et MEMBRE ci-dessus."
print("Expérience :", EXPERIMENT, "| run :", RUN_NAME)

Expérience : flexistaff_team-rocket | run : carole_v1


## 2. Connexion à MLflow

MLflow est protégé par le portail : le client doit présenter vos identifiants (les mêmes que pour CVAT).
Ils sont lus dans le fichier JSON et passés par variables d'environnement : ils ne sont **jamais** affichés ni écrits dans le notebook.
Vous n'avez *pas* besoin des identifiants S3 : les artefacts transitent par le serveur MLflow.

> Le fichier d'identifiants contient vos mots de passe : ne le committez jamais et ne le laissez pas dans un notebook partagé.

In [6]:
import json
import os

import mlflow
import onnx
from mlflow import MlflowClient

with open(CREDENTIALS, encoding="utf-8") as f:
    creds = json.load(f)["mlflow"]
os.environ["MLFLOW_TRACKING_USERNAME"] = creds["username"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = creds["password"]

mlflow.set_tracking_uri(creds["tracking_uri"])

# Test de connexion : on liste les expériences visibles
try:
    experiences = MlflowClient().search_experiments(max_results=5)
    print("Connexion MLflow OK :", [e.name for e in experiences])
except Exception as e:
    raise RuntimeError(f"Connexion à MLflow impossible : {e}\n"
                       "Vérifiez le fichier d'identifiants (dépôt dans le panneau Fichiers, bon login).") from e

mlflow.set_experiment(EXPERIMENT)   # créée si elle n'existe pas encore
print("Expérience :", EXPERIMENT)

Connexion MLflow OK : ['flexistaff_team-rocket', 'flexistaff_None', 'flexistaff_team-aqua', 'flexistaff_team-magma']
Expérience : flexistaff_team-rocket


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Jeu de données : décompresser l'export CVAT et découper train / val

L'export CVAT (*Ultralytics YOLO Detection 1.0*) contient `data.yaml`, `images/` et `labels/`.
Ultralytics attend cette structure, avec **deux ensembles** : `train` (le modèle apprend) et `val` (on mesure sur des images qu'il n'a jamais vues) :

```
dataset/
├── data.yaml
├── images/{train,val}/   *.jpg
└── labels/{train,val}/   *.txt   (même nom que l'image)
```

Le code ci-dessous décompresse l'archive, retrouve les images et leurs fichiers `.txt`, les mélange puis les répartit ≈ 80 / 20.
Les **noms de classes sont lus dans le `data.yaml` de l'export** : ce sont les vôtres, ne les écrivez pas à la main.

> Avec si peu d'images, les scores varieront beaucoup d'un tirage à l'autre : ne sur-interprétez pas un écart de quelques points.

In [8]:
import random
import shutil
import zipfile
from pathlib import Path

import yaml

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
brut = Path("cvat_brut")
dataset = Path(DATASET_DIR)

# 1) Décompression de l'export CVAT (on repart d'un dossier propre)
for d in (brut, dataset):
    shutil.rmtree(d, ignore_errors=True)
with zipfile.ZipFile(CVAT_ZIP) as z:
    z.extractall(brut)

# 2) data.yaml de l'export : c'est lui qui donne les classes, dans l'ordre de CVAT
data_yamls = list(brut.rglob("data.yaml"))
assert data_yamls, "data.yaml introuvable : avez-vous exporté au format « Ultralytics YOLO Detection 1.0 » ?"
racine = data_yamls[0].parent
cfg = yaml.safe_load(data_yamls[0].read_text(encoding="utf-8"))
names = cfg["names"]
if isinstance(names, list):          # liste ou dictionnaire {0: nom, ...} : on normalise
    names = dict(enumerate(names))
names = {int(k): v for k, v in names.items()}

# 3) Images et annotations. CVAT reproduit l'arborescence du bucket :
#    images/train/<équipe>/<lot>/xxx.jpg  et  labels/train/<équipe>/<lot>/xxx.txt
images = sorted(p for p in (racine / "images").rglob("*") if p.suffix.lower() in IMG_EXT)
assert images, "Aucune image trouvée dans images/ de l'export."

def label_de(img):
    # le .txt a le même nom que l'image, dans labels/ (même sous-dossier)
    rel = img.relative_to(racine / "images").with_suffix(".txt")
    return racine / "labels" / rel

# 4) Découpage train / val (≈ 80 / 20), reproductible grâce à SEED
random.Random(SEED).shuffle(images)
n_train = min(max(1, round(len(images) * RATIO_TRAIN)), len(images) - 1)
parts = {"train": images[:n_train], "val": images[n_train:]}

def nom_unique(img):
    # « Team Galaxy/lot-1/img_001.jpg » -> « Team_Galaxy__lot-1__img_001.jpg » :
    # deux lots peuvent contenir des images de même nom, on garde le chemin pour ne rien écraser
    rel = img.relative_to(racine / "images")
    return "__".join(rel.parts).replace(" ", "_")

sans_label = 0
for nom_part, liste in parts.items():
    (dataset / "images" / nom_part).mkdir(parents=True, exist_ok=True)
    (dataset / "labels" / nom_part).mkdir(parents=True, exist_ok=True)
    for img in liste:
        nom = nom_unique(img)
        shutil.copy(img, dataset / "images" / nom_part / nom)
        lbl = label_de(img)
        dest = dataset / "labels" / nom_part / (Path(nom).stem + ".txt")
        if lbl.exists():
            shutil.copy(lbl, dest)
        else:
            dest.write_text("")      # image sans annotation : fichier vide
            sans_label += 1

# 5) Notre data.yaml (chemins absolus, mêmes classes que CVAT)
DATA_YAML = dataset / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(dataset.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": len(names),
    "names": names,
}, allow_unicode=True, sort_keys=False), encoding="utf-8")

print(f"{len(images)} images : {len(parts['train'])} train / {len(parts['val'])} val")
print("Classes :", names)
if sans_label:
    print(f"⚠ {sans_label} image(s) sans annotation : normal pour quelques images vides, "
          "anormal si c'est la majorité (job pas terminé, mauvais export ?)")
assert sans_label < len(images), "Aucune annotation trouvée : vérifiez l'export CVAT."

40 images : 32 train / 8 val
Classes : {0: 'Titre_calendrier', 1: 'Date_case', 2: 'Ressource', 3: 'Evenement'}


## 4. Entraînement YOLO dans un run MLflow (et export ONNX, étape 5)

YOLO (*You Only Look Once*) détecte tous les objets en une seule passe : pour chaque zone, une **boîte**, une **classe** et une **confiance**.
On repart de poids pré-entraînés (`yolov8s.pt`) plutôt que de zéro.

Dans un run MLflow, on range cinq sortes d'éléments :

| Élément d'un run | Ce qu'on y met |
|---|---|
| params | ce qu'on a *choisi* (modèle, epochs, taille d'image, version du dataset) |
| metrics | ce qu'on a *obtenu* (mAP50, précision, rappel) |
| artifacts | les fichiers *produits* (`best.pt`, courbes) : onglet *Artifacts* |
| logged models | le modèle *déployable* (ONNX), rattaché au run : onglet *Logged models* |
| tags | squad, version du dataset |

> **TODO 1 : choisissez vos hyperparamètres.** Indices :
> - `MODELE` : `yolov8n.pt` (nano, le plus rapide, bon pour démarrer), `yolov8s.pt` (small, bon compromis : notre choix par défaut), `m`/`l`/`x` sont inutiles avec si peu de données ;
> - `EPOCHS` : nombre de passages sur le jeu d'entraînement (50 est un bon ordre de grandeur) ;
> - `IMGSZ` : taille d'image en pixels (640 par défaut) ; elle sera **figée** dans le fichier ONNX.
>
> **TODO 2 : dans `log_metrics`, ajoutez le rappel.** Les métriques sont mesurées sur `val` : `res.box.map50` (mAP50), `res.box.mp` (précision moyenne), `res.box.mr` (**rappel moyen**, celui qui compte le plus).
>
> **TODO 3 : exportez en ONNX et rangez-le dans le run (étape 5).** `best.pt` sert à *entraîner* ; ONNX sert à *déployer* : léger (`onnxruntime` suffit, ni PyTorch ni Ultralytics), portable (CPU, GPU, mobile), indépendant de la version d'Ultralytics ; les noms de vos classes y sont rangés. C'est ce format que télécharge le notebook de test OCR (jour 2) et que charge l'application (jour 3). Contreparties : la taille d'image est figée à l'export, et le post-traitement (seuil de confiance, suppression des doublons) est à refaire côté application.
> Dans le bloc `with`, à la fin (le run doit être encore ouvert) : (1) `onnx_path = YOLO(best_pt).export(format="onnx", imgsz=IMGSZ)` ; (2) charger le fichier avec `onnx.load(onnx_path)` ; (3) le ranger dans le run avec `mlflow.onnx.log_model(<modèle chargé>, name="onnx", save_as_external_data=False)`.

In [3]:
MODELE = "yolov8n.pt"   # TODO 1 : poids de départ, ex. "yolov8s.pt"
EPOCHS = 50   # TODO 1 : nombre d'epochs, ex. 50
IMGSZ = 640   # TODO 1 : taille d'image, ex. 640
BATCH = 8     # images par lot (réduisez à 8 en cas d'erreur de mémoire GPU)

assert None not in (MODELE, EPOCHS, IMGSZ), "Complétez MODELE, EPOCHS et IMGSZ (TODO 1)."

In [9]:
from ultralytics import YOLO, settings

# Le suivi MLflow est fait à la main ci-dessous. On coupe celui intégré à Ultralytics, qui sinon
# relogue ses propres paramètres dans le même run (conflit) et change l'expérience courante.
settings.update({"mlflow": False})

model = YOLO(MODELE)   # poids pré-entraînés : on ne repart pas de zéro

with mlflow.start_run(run_name=RUN_NAME) as run:                       # <membre>_<version>
    RUN_ID = run.info.run_id
    mlflow.set_tags({"squad": SQUAD, "dataset": DATASET_VERSION})
    mlflow.log_params({"model": MODELE.replace(".pt", ""), "epochs": EPOCHS, "imgsz": IMGSZ,
                       "batch": BATCH, "n_train": len(parts["train"]), "n_val": len(parts["val"]),
                       "classes": ",".join(names.values())})

    res = model.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED)

    # Métriques mesurées sur le jeu val
    mlflow.log_metrics({
        "mAP50": res.box.map50, # mAP50 = mesure globale de la qualité des détections
        "precision": res.box.mp, # Précision = proportion de détections correctes parmi celles proposées
        "recall": res.box.mr,   # Rappel (recall) : proportion des zones réellement présentes que YOLO a retrouvées
        # TODO 2 : ajoutez ici le rappel (clé "recall", valeur res.box.mr)
    })

    best_pt = model.trainer.best    # runs/detect/train*/weights/best.pt
    mlflow.log_artifact(str(best_pt))
    mlflow.log_artifact(str(Path(model.trainer.save_dir) / "results.png"))   # courbes d'entraînement

    # ---- Étape 5 : export ONNX, rangé dans le run ----
    onnx_path = YOLO(best_pt).export(format="onnx", imgsz=IMGSZ) # export : convertit le modèle entraîné au format ONNX.
    modele_onnx = onnx.load(onnx_path)  # onnx.load : charge le fichier ONNX obtenu.
    mlflow.onnx.log_model(modele_onnx, name="onnx", save_as_external_data=False) # mlflow.onnx.log_model : enregistre ce modèle dans MLflow pour pouvoir le réutiliser.
    # TODO 3 : (1) exporter en ONNX, (2) charger le fichier, (3) mlflow.onnx.log_model(..., name="onnx", save_as_external_data=False)

print("Run :", RUN_NAME, "| run_id :", RUN_ID)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, 

## 5. Vérifier l'export ONNX dans le run

L'export lui-même se fait dans la cellule d'entraînement (TODO 3). Cette cellule contrôle que le modèle ONNX est bien rangé dans le run : avec MLflow 3, il n'est pas dans l'onglet *Artifacts* mais dans *Logged models*.
Si elle échoue, complétez le TODO 3 puis relancez l'entraînement : un run terminé ne se complète pas proprement (voir l'avertissement de la section 7), ce sera un **nouveau run** (passez `VERSION` à `v2` plus haut).

Elle contrôle aussi le **format de sortie** du fichier ONNX : l'application et l'API attendent `[1, 4 + nb_classes, N]` (4 coordonnées de boîte, un score par classe, N candidats avant la suppression des doublons). YOLOv8, YOLO11 et YOLO26 donnent ce format avec l'export du TODO 3 ; pas d'option `end2end` ni `nms` à l'export.

In [ ]:
# Vérification : le run contient-il bien best.pt (artefact) et le modèle ONNX (logged model) ?
client = MlflowClient()
fichiers = [a.path for a in client.list_artifacts(RUN_ID)]
print("Artefacts du run :", fichiers)

# MLflow 3 : le modèle ONNX n'est pas un artefact du run, c'est un « logged model » qui lui est rattaché
experience = client.get_run(RUN_ID).info.experiment_id
modeles = [m.name for m in mlflow.search_logged_models(experiment_ids=[experience], output_format="list")
           if m.source_run_id == RUN_ID]
print("Modèles du run :", modeles)

assert "onnx" in modeles, "Modèle ONNX absent du run : complétez le TODO 3 puis relancez l'entraînement."
print("OK : ONNX présent dans le run.")

# Format de sortie attendu par l'application (CalEvent) et par l'API : [1, 4 + nb_classes, N]
onnx_local = Path(best_pt).with_suffix(".onnx")   # l'export écrit best.onnx à côté de best.pt
sortie = onnx.load(str(onnx_local)).graph.output[0]
forme = [d.dim_value or d.dim_param for d in sortie.type.tensor_type.shape.dim]
attendu = 4 + len(names)
print("Sortie ONNX :", forme)
assert len(forme) == 3 and forme[1] == attendu, (
    f"Sortie ONNX {forme} : l'application attend [1, {attendu}, N] (4 coordonnées + {len(names)} classes).\n"
    "Exportez avec la commande du TODO 3, sans option end2end ni nms (YOLOv8, YOLO11 ou YOLO26) ; "
    "une sortie [1, 300, 6] vient d'un export « sans NMS » (YOLOv10, ou YOLO26 avec end2end=True).")
print("OK : format de sortie compatible avec l'application.")

## 6. Regarder les résultats

Retrouvez le run dans l'interface MLflow (lien du portail) : paramètres, métriques, `best.pt` (onglet *Artifacts*) et modèle ONNX (onglet *Logged models*).

> **TODO 4 : quelle métrique regarder en premier ?** Complétez `METRIQUE_CLE` avec la métrique qui compte le plus pour FlexiStaff (une case oubliée est un créneau perdu). Choix possibles : `"precision"`, `"recall"`, `"mAP50"`. Indice : relisez l'encadré du début.

In [ ]:
METRIQUE_CLE = None   # TODO 4 : "precision", "recall" ou "mAP50"
assert METRIQUE_CLE is not None, "Complétez METRIQUE_CLE (TODO 4)."

metriques = client.get_run(RUN_ID).data.metrics
print("Métriques sur val :")
for nom, val in metriques.items():
    if nom in ("mAP50", "precision", "recall"):
        marque = "  <-- à regarder en premier" if nom == METRIQUE_CLE else ""
        print(f"  {nom:<10} {val:.3f}{marque}")

Regarder des prédictions à l'œil reste le meilleur premier test. Ci-dessous : le modèle entraîné sur quelques images de `val` (jamais vues à l'entraînement).
Les boîtes manquantes sont des **oublis** (rappel trop bas) ; les boîtes en trop, des **fausses alarmes** (précision trop basse).

In [ ]:
import matplotlib.pyplot as plt

best = YOLO(str(best_pt))
echantillon = sorted((dataset / "images" / "val").glob("*"))[:4]

fig, axes = plt.subplots(1, len(echantillon), figsize=(5 * len(echantillon), 6), squeeze=False)
for ax, img in zip(axes[0], echantillon):
    r = best.predict(str(img), conf=0.25, imgsz=IMGSZ, verbose=False)[0]
    ax.imshow(r.plot()[..., ::-1])          # BGR -> RGB
    ax.set_title(f"{img.name} ({len(r.boxes)} détections)", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. Enregistrer le champion (Registry)

Chaque membre a son run `<membre>_v1` avec son modèle ONNX. **Comparez-les dans MLflow** (sélectionnez les runs puis *Compare*) et choisissez le meilleur avec le critère de promotion de la formation :

1. le **rappel** d'abord : une case oubliée, c'est un créneau perdu ;
2. le **mAP50** pour départager deux modèles proches ;
3. c'est le **chef de squad** qui décide.

> **Attention : le jour 1, le premier modèle devient `champion`.** Aucun champion n'existe encore, donc le meilleur `v1` de la squad prend cet alias.
> Dès le jour 2, **ne remplacez plus** le champion directement : vos nouveaux modèles reçoivent l'alias `challenger`, on les compare au champion, et le chef décide de la promotion.
> Seul le chef enregistre le modèle de la squad ; les autres peuvent sauter cette cellule.

On utilise des **alias** (`champion`, `challenger`), pas les anciens « stages ». Le registre contient ensuite une version du modèle `flexistaff_<squad>`, que chargent le notebook de test OCR (jour 2) et l'application (jour 3).

> **TODO 5 : renseignez `RUN_ID` et posez l'alias.** `RUN_ID` est l'identifiant du run choisi (page du run dans MLflow, ou la variable `RUN_ID` affichée à l'entraînement). Puis, avec `set_registered_model_alias(nom, alias, version)`, donnez l'alias `"champion"` à la version créée.

> N'utilisez **pas** `with mlflow.start_run(run_id=...)` pour compléter un run terminé : MLflow le rouvre et réécrit son heure de fin, la durée du run devient fausse. L'enregistrement ci-dessous ne modifie pas le run.

In [ ]:
RUN_ID_CHOISI = None   # TODO 5 : identifiant du run choisi, ex. RUN_ID (le vôtre) ou celui d'un coéquipier
ALIAS = None           # TODO 5 : "champion" aujourd'hui (jour 1) ; "challenger" à partir du jour 2

assert RUN_ID_CHOISI and ALIAS, "Complétez RUN_ID_CHOISI et ALIAS (TODO 5)."

mv = mlflow.register_model(f"runs:/{RUN_ID_CHOISI}/onnx", MODEL_NAME)
MlflowClient().set_registered_model_alias(MODEL_NAME, ALIAS, mv.version)
print(f"{MODEL_NAME} version {mv.version} -> alias « {ALIAS} »")

Pour récupérer le fichier ONNX en local (vérification, ou usage hors MLflow) :

In [ ]:
mlflow.artifacts.download_artifacts(f"models:/{MODEL_NAME}@{ALIAS}", dst_path="champion")
print(sorted(str(p) for p in Path("champion").rglob("*.onnx")))   # -> champion/model.onnx

### Fiche du modèle : vos classes → les rôles de CalEvent (chef de squad)

CalEvent ne connaît pas **vos** noms de classes : il lit des **rôles** (`calendar`, `month_header`, `day_header_row`,
`date_cell`, `event`, `resource_header`). Un tag `class_map` posé sur le modèle enregistré lui dit lequel est lequel ;
il le relit tout seul au rechargement (↻), pour le champion comme pour les futurs challengers.

Format : `votre_classe=rôle`, séparés par des virgules (ex. `jour=date_cell,creneau=event,agent=resource_header`).
Une classe qui porte déjà le nom d'un rôle n'a pas besoin d'être listée ; une classe sans rôle est simplement ignorée.

In [ ]:
CLASS_MAP = None   # ex. "jour=date_cell,creneau=event,agent=resource_header" ; None = vos classes portent déjà les noms des rôles

print("Vos classes :", list(best.names.values()))
if CLASS_MAP:
    MlflowClient().set_registered_model_tag(MODEL_NAME, "class_map", CLASS_MAP)
    print(f"Tag class_map posé sur {MODEL_NAME} : {CLASS_MAP}")
else:
    print("Pas de tag class_map : CalEvent utilisera les noms de classes tels quels.")

## Checklist de fin de journée

- [ ] Un entraînement s'est terminé sans erreur
- [ ] Ce run apparaît dans MLflow (paramètres, métrique, `best.pt`, modèle ONNX, tags squad et dataset)
- [ ] Vous avez regardé une prédiction (idéalement sur une vraie photo)
- [ ] Le meilleur `v1` de la squad est enregistré avec l'alias `champion` (et son tag `class_map` si vos classes ont leurs propres noms)
- [ ] Le code d'entraînement (ce notebook, **sans le fichier d'identifiants**) est poussé dans `ai_training/` du fork de la squad